In [1]:
import pandas as pd

from freeze_thaw.data_preparation.general import align_timestamps
from freeze_thaw.config import config as c
from freeze_thaw.config import StationName
from freeze_thaw.data_preparation.feature_engineering import create_lagged_features, cyclical_encoding
from freeze_thaw.data_preparation.splitting import train_test_split
from freeze_thaw.modeling.train import train_model

# Model Development

The goal is to train a lightGBM model and determine whether to use lagged features or not based on a time series cross-validation on 80% of the data. The rest of the data is reserved for evaluating the model trained on the ASCAT dataset to the ERA5 predictions.

---

## Table of Contents

1. **Setup**
    - *1.1 Variables*
    - *1.2 Functions*
2. **ISMN Stations**
    - *2.1 Aberdeen-35-WNW*
    - *2.2 Jamestown-38-WSW*
    - *2.3 Gobblers Knob*
    - *2.4 Nenana*
    - *2.5 L23*
    - *2.6 L38*
    - *2.7 NST-07*
    - *2.8 NST-09*
    - *2.9 SOD012*
    - *2.10 SOD103*

---

## 1. Setup

### 1.1 Variables

In [2]:
lags = [1, 2, 3, 4, 5]
label_map = {
    c.CLASSES[0]: 0,
    c.CLASSES[1]: 1,
    c.CLASSES[2]: 2,
}
train_size = 0.8

### 1.2 Functions

In [3]:
def process_df(df: pd.DataFrame, label_encoding: dict[str, int], lagged_features: bool,
               lags: list[int] | None = None) -> pd.DataFrame:
    """
    Prepare df for training of lightGBM model. Class labels will be converted to integers, which is required for model input.
    :param df: pd.DataFrame
    :param label_encoding: dict mapping c.CLASSES to int
    :param lagged_features: whether to use lagged features
    :param lags: list of lags to create
    :return: processed df
    """
    df_copy = df.copy()

    if lagged_features:
        df_copy = create_lagged_features(df_copy, lags)
    df_copy = cyclical_encoding(df_copy)

    null_count = df_copy[df_copy["class"].isnull()].shape[0]
    print(f"Dropping {null_count} rows with no class label.")
    df_copy = df_copy.dropna(subset=["class"])

    df_copy['class'] = df_copy['class'].map(label_encoding)

    return df_copy

## 2. ISMN Stations

### 2.1 Aberdeen-35-WNW

#### Create train/test split

In [4]:
aberdeen_ascat_df, _ = align_timestamps(StationName.ABERDEEN, c.CLEANED_DATA_PATH)

aberdeen_ascat_train, _ = train_test_split(aberdeen_ascat_df, train_size)

#### Without lagged features

In [5]:
aberdeen_ascat_unlagged_train = process_df(aberdeen_ascat_train, lagged_features=False, label_encoding=label_map, lags=lags)
aberdeen_unlagged_train_result = train_model(aberdeen_ascat_unlagged_train, n_splits=5, label_encoding=label_map)

print(f"Average macro F1 score: {round(aberdeen_unlagged_train_result.average_macro_f1, 3)}")
print(f"Average transition F1 score: {round(aberdeen_unlagged_train_result.average_transition_f1, 3)}")

Dropping 243 rows with no class label.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000264 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 758
[LightGBM] [Info] Number of data points in the train set: 2129, number of used features: 6
[LightGBM] [Info] Start training from score -1.554160
[LightGBM] [Info] Start training from score -1.799776
[LightGBM] [Info] Start training from score -0.472732
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000280 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 759
[LightGBM] [Info] Number of data points in the train set: 4253, number of used features: 6
[LightGBM] [Info] Start training from score -1.572055
[LightGBM] [Info] Start training from score -1.739315
[LightGBM] [Info] Start training from score 

#### With lagged features

In [6]:
aberdeen_ascat_lagged_train = process_df(aberdeen_ascat_train, lagged_features=True, label_encoding=label_map, lags=lags)
aberdeen_lagged_train_result = train_model(aberdeen_ascat_lagged_train, n_splits=5, label_encoding=label_map)

print(f"Average macro F1 score: {round(aberdeen_lagged_train_result.average_macro_f1, 3)}")
print(f"Average transition F1 score: {round(aberdeen_lagged_train_result.average_transition_f1, 3)}")

Dropping 243 rows with no class label.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000306 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2134
[LightGBM] [Info] Number of data points in the train set: 2124, number of used features: 16
[LightGBM] [Info] Start training from score -1.551809
[LightGBM] [Info] Start training from score -1.797425
[LightGBM] [Info] Start training from score -0.474155
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000283 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2153
[LightGBM] [Info] Number of data points in the train set: 4248, number of used features: 16
[LightGBM] [Info] Start training from score -1.570878
[LightGBM] [Info] Start training from score -1.738138
[LightGBM] [Info] Start training from sc